In [1]:
import torch

In [142]:
class Linear ( torch.nn.Module ):
    def __init__ ( self, fan_in, fan_out, device = None, dtype = None ) :
        super().__init__()
        sigma =  ( 2 / ( fan_in + fan_out ) )**0.5
        self.M  = torch.nn.Parameter (torch.empty ( ( fan_in, fan_out ), device = device, dtype = dtype )  )
        torch.nn.init.trunc_normal_ (self.M, std= sigma , a=-3. * sigma , b=3.0 * sigma) 

    def forward ( self, xin ) :
        return xin @ self.M

class Embedding ( torch.nn.Module ):
    def __init__ ( self, V, C , device = None, dtype = None ) :
        super().__init__()
        self.Cmap = torch.nn.Parameter ( torch.empty ( ( V, C ) , device = device, dtype = dtype ) )
        torch.nn.init.trunc_normal_ (self.Cmap, std= 1. , a=-3. , b=3.0) 

    def forward ( self, xin ) :
        return self.Cmap [ xin ]

class RmsNorm (  torch.nn.Module ):
    def __init__ ( self, C , eps = 1e-6, device = None, dtype = None ) :
        super().__init__()
        self.gamma = torch.nn.Parameter ( torch.ones( C , device = device, dtype = dtype )  )
        self.eps = eps
        
    def forward ( self, xin ) :
        y = xin *xin
        sigma2 = torch.mean ( y, dim = -1, keepdim = True ) 
        out = xin / ( sigma2**0.5 + self.eps ) * self.gamma
        return out

def softmax ( xin , dim = -1) :
    shifted  = xin - torch.max ( xin, dim = dim , keepdim=True ).values
    expsum = torch.sum ( shifted.exp(), dim = dim , keepdim= True )
    return shifted.exp()/ expsum 




def Attention ( Q, K , V ) :
    T, dk = K.shape[-2:]
    masked = torch.triu ( torch.ones ( ( T, T ) , dtype = torch.bool), diagonal = 1 )  
    qk = Q@ torch.transpose ( K, -2,-1 )
    qk = qk.masked_fill ( masked , float('-inf')) 
    out = softmax ( qk * dk**-0.5 )  @ V
    return out


class RoPE ( torch.nn.Module ):
    def __init__ ( self,Tmax, dk, theta ,  device = None, dtype = None ) :
        super().__init__()
        dkh = dk//2
        thetalist  = torch.empty ( ( Tmax, dkh ) ,  device = None, dtype = None ) 
        for ii in range ( Tmax ):
            thetalist [ii  ] = torch.tensor ( [ ii / theta** ( kk / dkh ) for kk in range ( dkh ) ] )
        st  = thetalist.sin()
        ct  = thetalist.cos()
        self.register_buffer ('costheta',ct ,  persistent = False ) 
        self.register_buffer ('sintheta',st ,  persistent = False )  # why using buffer and why persistent = False?

    def forward ( self, xin , positions = None) :
        x1 = xin [..., ::2]
        x2 = xin[...,1::2]

        T = xin.shape[-2]

        if positions is not None :
            ct = self.costheta [positions]
            st  = self.sintheta [positions]
        
        else :
            ct = self.costheta [:T]
            st  = self.sintheta [:T]
        
        
        x1new = x1 * ct  - x2* st
        x2new = x1 * st  + x2* ct
        xout = torch.empty_like ( xin ) 
        xout [..., ::2] = x1new
        xout [..., 1::2] = x2new
        return xout
        
        
            
        
        


class MultiHeadAttention ( torch.nn.Module ):
    def __init__ ( self, C ,  Tmax,  nH, theta = None,  device = None, dtype = None ) :
        super().__init__()
        dk = C //nH
        dv = C//nH
        self.Mq = Linear ( C, dk * nH,  device = device, dtype = dtype )
        self.Mk = Linear ( C, dk * nH,  device = device, dtype = dtype )
        self.Mv = Linear ( C, dv * nH,  device = device, dtype = dtype )
        self.Mo  = Linear ( dv * nH, C,  device = device, dtype = dtype )
        self.theta = theta
        self.nH = nH
        self.dk = dk
        self.dv =dv
        self.rope = RoPE( Tmax, dk, theta )  
        
    def forward ( self, xin ) :
        Q = self.Mq ( xin )
        K =  self.Mk ( xin )
        V = self.Mv ( xin )
        kvlist = []
        for ii in range ( self.nH ) :
            Qi = Q [..., ii* self.dk: ( ii+1 ) *self.dk ]
            Ki  = K [..., ii* self.dk: ( ii+1 )* self.dk ]

            Qi = self.rope ( Qi ) 
            Ki = self.rope ( Ki ) 
            
            Vi  = V [..., ii* self.dv: ( ii+1 ) *self.dv ]
            
            kv = Attention ( Qi, Ki, Vi )
            kvlist.append ( kv )
        out = torch.cat ( kvlist, dim =-1)
        return self.Mo ( out )
            


class FFN ( torch.nn.Module ):
    def __init__ ( self, C ,  dff ,  device = None, dtype = None ) :
        super().__init__()
        self.W1 = Linear ( C, dff ,  device = device, dtype = dtype )
        self.W3 = Linear ( C, dff ,  device = device, dtype = dtype )
        self.W2 = Linear (  dff, C ,  device = device, dtype = dtype )

    def forward ( self, xin ) :
        x1 = self.W1 (xin)
        x3  = self.W3 ( xin ) 

        out =  self.W2( x1 * torch.sigmoid ( x1) * x3 ) 

        return out


class TransformerBlock ( torch.nn.Module ):
    def __init__ ( self, C ,  Tmax, nH, dff, theta = None, eps= 1e-6,  device = None, dtype = None ) :
        super().__init__()
        self.ln1 = RmsNorm ( C, eps = eps, device = device, dtype = dtype )
        self.ln2 = RmsNorm ( C, eps = eps, device = device, dtype = dtype )
        self.mha = MultiHeadAttention ( C ,  Tmax, nH, theta= theta,  device = device, dtype = dtype )
        self.ffn = FFN ( C, dff ,  device = device, dtype = dtype ) 

    def forward ( self, xin ) :
        x = self.ln1 ( xin )
        x = xin + self.mha ( x )

        y = self.ln2 ( x )
        y = x + self.ffn ( y ) 
        return y 


class TransformerLM ( torch.nn.Module ):
    def __init__ ( self,V, C ,  Tmax, L,  nH, dff, theta = None, eps= 1e-6,  device = None, dtype = None ) :
        super().__init__()
        self.embd  = Embedding (  V, C , device = device, dtype = dtype ) 
        self.blocks = torch.nn.ModuleList ( [ TransformerBlock  ( C ,  Tmax,  nH, dff,  theta= theta,  device = device, dtype = dtype ) 
                            for _ in range  ( L ) ])
        self.ln =  RmsNorm ( C, eps = eps, device = device, dtype = dtype )
        self.linear  = Linear ( C, V,  device = device, dtype = dtype ) 


    def forward ( self, xin ) :
        x = self.embd ( xin )
        for bb in self.blocks :
            x = bb ( x) 
        x = self.ln ( x) 
        return self.linear (x)
            


def CrossEntropy ( logits, ypred ) :
    x = logits.view ( -1, logits.shape[-1] )
    y = ypred.view ( -1) 
    shifted = x - torch.max ( x, dim = -1, keepdim = True ). values
    xexpsum = torch.sum ( shifted.exp(), dim = -1 ) 

    out = - shifted [ torch.arange( len(y)), y ] + xexpsum.log()
    return out.mean()

        

In [35]:
cl1= Linear ( 3,3)
a = torch.randn ( ( 4,3 ) )
cl1(a).shape

torch.Size([4, 3])

In [107]:
V = 16
T = 8
C = 8
B = 4
dff = 32
L=6

Theta = 1000


H = 2
tokenI = torch.randint ( V , ( B, T ) )
tokenI

tensor([[ 1, 12,  9, 15,  5,  4, 10,  5],
        [15,  1,  0,  3,  5,  5,  6, 15],
        [ 8,  2,  6,  4,  4, 15, 13, 12],
        [ 1, 10,  2,  8,  8,  7, 13, 11]])

In [110]:
embd = Embedding( V, C )
x = embd ( tokenI)
rms= RmsNorm ( C )
x= rms (x)
print (x.shape )
mht = MultiHeadAttention ( C ,  T, H, Theta)
x =mht ( x)
x.shape
ffn = FFN ( C, dff )
x= ffn ( x)

tblock = TransformerBlock ( C ,  T, H, dff,Theta)
x= tblock ( x)
tlm = TransformerLM ( V, C ,  T, L, H, dff,Theta)
x = tlm (tokenI)
x.shape

torch.Size([4, 8, 8])


torch.Size([4, 8, 16])

In [49]:
torch.triu ( torch.ones ( ( 3,3 ) , dtype = torch.bool), diagonal=1 )  

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [111]:
## load data

import numpy as np

file_ids = "assignment1-basics/data/tinystories_valid.npy"
data = np.load ( file_ids, mmap_mode="r" )

In [144]:
V= 10000
T = 256
C = 512
dff = round ( 8 * C / 3 / 64 ) * 64
Theta = 10000
L = 4
H = 16
B= 8

In [145]:
tclass = TransformerLM ( V, C ,  T, L, H, dff,Theta)
params  = tclass.parameters()
lr = 1e-3
wdecay = 0.01
betas=(0.9, 0.999)
adamW  = torch.optim.AdamW(params, lr=lr, betas=betas, weight_decay= wdecay,)

In [ ]:
lossi =[]
runNum = 100
xinput = torch.empty (( B, T ), dtype = torch.long ) 
ypred = torch.empty (( B, T ), dtype = torch.long ) 
datasize = len ( data ) 
for ii in range ( runNum ) :
    adamW.zero_grad()
    Blist = torch.randint ( datasize - T, ( B, ) ) 
    for kk in range ( B  ):
        xinput[kk] = torch.tensor( data [ Blist[kk] : Blist[kk] + T ])
        ypred [kk] = torch.tensor ( data[ Blist[kk] +1 : Blist[kk] + T+1 ])

    logits = tclass ( xinput )
    loss = CrossEntropy ( logits, ypred ) 
    loss.backward()

    adamW.step ()
    print ( f"step {ii}, loss = {loss.item() }")
    lossi.append( loss.item())
    if ii ==10 :
        break

step 0, loss = 9.243792533874512
step 1, loss = 8.68822193145752
step 2, loss = 7.901496410369873


In [ ]:
a